# Master's Thesis Run Notebook

This notebook is part of the **Master's Thesis (MSc Dissertation)**: **Fast Simulation of Neutrino Oscillations in Matter**.

**Author**  
Juan Ramon Diaz Santos <diazjuan@alumni.uv.es>

**Supervisors**  
Roberto Ruiz de Austri Bazan <rruiz@ific.uv.es>  
Michele Lucente <michele.lucente@unibo.it>

**Date**  
June 2026


# Atmosphere 2: Detector Flux Propagation

Propagate production fluxes through atmosphere and Earth to produce detector-level flux files for each particle-angle pair.


## 1. Libraries

This section imports Python standard-library modules, PyTorch, repository helpers, and the TPeanuts APIs required by the run. It also locates the repository root so the notebook can be executed from Jupyter without relying on the script `__file__` variable.


In [1]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path
from typing import Dict, Optional

import torch

from tpeanuts.core.pmns import PMNS
from tpeanuts.io.io_earth import load_earth_density_from_csv
from tpeanuts.flux_propagation.pipeline_atmosphere import integrate_height_and_sum_flavours, integrate_initial_and_surface_fluxes, propagate_atmosphere_coherent, propagate_earth_coherent, select_particle_angle_flux
from tpeanuts.io.io_flux_propagation import build_detector_flux_path, save_detector_flux_result
from tpeanuts.io.io_atmosphere import load_directory
from tpeanuts.util.torch_util import _default_device, resolve_device
from tpeanuts.util.type import _as_tensor

from tpeanuts.util.notebooks import find_repo_root
HERE = Path.cwd().resolve()
PACKAGE_DIR = find_repo_root(HERE, folder="analysis")
print(f"PACKAGE_DIR = {PACKAGE_DIR}")


PACKAGE_DIR = G:\Mi unidad\03.Codigo\034.TFM.UV\Tpeanuts


## 2. Paths and Configuration

This section centralizes every filesystem path and all run parameters before any computation is started. Keeping these values together makes the run reproducible and avoids hidden output locations.


### 2.1. Paths

All outputs are rooted at `DEFAULT_OUTPUT_ROOT` unless the environment variable `TPEANUTS_OUTPUT_ROOT` is set. Derived folders are created from that single root, with atmospheric production outputs written under `OUTPUT_DATA_ROOT / "atmosphere"`.


In [2]:
DEFAULT_OUTPUT_ROOT = Path(r"V:\output")
OUTPUT_ROOT = Path(os.environ.get("TPEANUTS_OUTPUT_ROOT", DEFAULT_OUTPUT_ROOT))

OUTPUT_DATA_ROOT = Path(OUTPUT_ROOT / "data")
OUTPUT_ANALYSIS_ROOT = Path(OUTPUT_ROOT / "analysis")
OUTPUT_BENCHMARK_ROOT = Path(OUTPUT_ROOT / "benchmark")
OUTPUT_TEST_ROOT = Path(OUTPUT_ROOT / "test")

OUTPUT_DATA_ATMOSPHERE = Path(OUTPUT_DATA_ROOT / "atmosphere")
OUTPUT_DATA_SOLAR = Path(OUTPUT_DATA_ROOT / "solar")
OUTPUT_DATA_EXTERNAL = Path(OUTPUT_DATA_ROOT / "external")

OUTPUT_ANALYSIS_ATMOSPHERE = Path(OUTPUT_ANALYSIS_ROOT / "atmosphere")
OUTPUT_ANALYSIS_SOLAR = Path(OUTPUT_ANALYSIS_ROOT / "solar")
OUTPUT_ANALYSIS_EXTERNAL = Path(OUTPUT_ANALYSIS_ROOT / "external")

OUTPUT_DATA_MCEQ = Path(OUTPUT_DATA_EXTERNAL / "mceq")
OUTPUT_DATA_HONDA = Path(OUTPUT_DATA_EXTERNAL / "honda")

OUTPUT_ATMOSPHERE_ROOT = Path(OUTPUT_DATA_ATMOSPHERE)
OUTPUT_MCEQ_ROOT = Path(OUTPUT_ATMOSPHERE_ROOT / "mceq")
INPUT_DIR = str(OUTPUT_MCEQ_ROOT / "diff_flux_QGSJETII04_PolyGonato_ICECUBE")

DETECTOR_ROOT = Path(OUTPUT_ATMOSPHERE_ROOT / "detector")
OUTPUT_DIR = str(DETECTOR_ROOT / "detector_flux_QGSJETII04_PolyGonato_ICECUBE")

OUTPUT_FILENAME = "detector_flux.pt"
EARTH_DENSITY_FILE = str(PACKAGE_DIR / "data" / "density" / "earth_density.csv")
DETECTOR_ROOT.mkdir(parents=True, exist_ok=True)


## 2.2. Parameters and System Configuration

This subsection defines the system configuration used by the run. The parameters control the physical model, angular and numerical grids, output precision, runtime device, batching strategy, and overwrite/skip behavior. Adjust these values before executing the run cells when producing a different dataset. The expected result is a consistent set of validated configuration objects and reproducible output paths. Possible problems include unavailable external data, missing MCEq models, excessive memory use from large chunks, or accidental reuse of existing files when `SKIP_EXISTING` is enabled.


### Data products

These parameters define the data products written by this run. Review them before executing production cells, especially when changing datasets, precision, or runtime size.


In [3]:
OVERWRITE = False

SAVE_DTYPE = torch.float32


### Physics

These parameters define the oscillation physics used by this run. Review them before executing production cells, especially when changing datasets, precision, or runtime size.


In [4]:
DM21_EV2 = 7.42e-5
DM3L_EV2 = 2.517e-3

THETA12 = 0.59
THETA13 = 0.15
THETA23 = 0.78

DELTA_CP = 1.20

DETECTOR_DEPTH_M = 1000.0

MATTER_IN_atmosphere = True

REUNITARIZE_earth = False


### Runtime and batching

These parameters control runtime behavior and batching for this run. Review them before executing production cells, especially when changing datasets, precision, or runtime size.


In [5]:
LOAD_DEVICE = "cpu"
PROPAGATION_DEVICE = _default_device
COMPUTE_DTYPE = torch.float64
LOAD_DTYPE = torch.float64

ENERGY_CHUNK_SIZE: Optional[int] = 8

HEIGHT_MAX_COUNT: Optional[int] = None
ENERGY_MAX_COUNT: Optional[int] = None
ANGLE_MAX_COUNT: Optional[int] = None

atmosphere_STEPS = 120
TRAJECTORY_STEPS = 120

PARTICLES: Optional[tuple[str, ...]] = None

SKIP_EXISTING = True

DEBUG = True


## 3. Helper functions

The following cells define the helper functions used by the run. Each function is kept in its own code cell so it can be inspected, edited, and rerun independently during interactive validation.


### Function: `evenly_spaced_indices`

This helper prepares or transforms part of the run state used by `main()`. The expected result is that it returns validated tensors, configuration objects, paths, or partial results without writing unrelated files.


In [6]:
def evenly_spaced_indices(n_items: int, max_count: Optional[int] = None) -> torch.Tensor:
    if max_count is None or max_count >= n_items:
        return torch.arange(n_items, dtype=torch.long)

    return torch.linspace(
        0,
        n_items - 1,
        int(max_count),
        dtype=torch.float64,
    ).round().to(torch.long).unique()


### Function: `chunk_indices`

This helper prepares or transforms part of the run state used by `main()`. The expected result is that it returns validated tensors, configuration objects, paths, or partial results without writing unrelated files.


In [7]:
def chunk_indices(indices: torch.Tensor, chunk_size: Optional[int]):
    indices = indices.reshape(-1).to(dtype=torch.long)

    if chunk_size is None or chunk_size <= 0:
        yield indices
        return

    for start in range(0, indices.numel(), int(chunk_size)):
        yield indices[start:start + int(chunk_size)]


### Function: `infer_antinu`

This helper prepares or transforms part of the run state used by `main()`. The expected result is that it returns validated tensors, configuration objects, paths, or partial results without writing unrelated files.


In [8]:
def infer_antinu(particle: str) -> bool:
    key = str(particle).lower()
    return "anti" in key or "bar" in key


### Function: `angle_value`

This helper prepares or transforms part of the run state used by `main()`. The expected result is that it returns validated tensors, configuration objects, paths, or partial results without writing unrelated files.


In [9]:
def angle_value(selected: Dict[str, object]) -> tuple[Optional[float], float]:
    alpha = selected.get("alpha_deg", None)
    theta = selected.get("theta_deg", None)

    alpha_value = None if alpha is None else float(alpha)
    theta_value = float(theta)

    return alpha_value, theta_value


### Function: `thin_selected`

This helper prepares or transforms part of the run state used by `main()`. The expected result is that it returns validated tensors, configuration objects, paths, or partial results without writing unrelated files.


In [10]:
def thin_selected(
    selected: Dict[str, object],
    *,
    energy_indices: torch.Tensor,
    height_indices: torch.Tensor,
) -> Dict[str, object]:
    thinned = dict(selected)

    E = selected["E_grid_GeV"][energy_indices]
    h = selected["h_grid_km"][height_indices]
    phi_Eh = selected["phi_Eh"][energy_indices][:, height_indices]

    thinned["E_grid_GeV"] = E
    thinned["h_grid_km"] = h
    thinned["phi_Eh"] = phi_Eh
    thinned["phi_E_theta"] = torch.trapezoid(phi_Eh, x=h, dim=-1)

    if selected.get("f_Eh", None) is not None:
        thinned["f_Eh"] = selected["f_Eh"][energy_indices][:, height_indices]

    return thinned


### Function: `propagate_particle_angle`

This helper prepares or transforms part of the run state used by `main()`. The expected result is that it returns validated tensors, configuration objects, paths, or partial results without writing unrelated files.


In [11]:
def propagate_particle_angle(
    *,
    selected: Dict[str, object],
    pmns: PMNS,
    density,
    dm21: torch.Tensor,
    dm3l: torch.Tensor,
    antinu: bool,
    energy_indices: torch.Tensor,
    height_indices: torch.Tensor,
    device: torch.device,
    dtype: torch.dtype,
) -> Dict[str, torch.Tensor]:
    detector_flux_chunks = []
    surface_flux_chunks = []
    initial_flux_chunks = []
    detector_probability_chunks = []
    surface_probability_chunks = []
    source_flux_chunks = []
    E_chunks = []

    for chunk in chunk_indices(energy_indices, ENERGY_CHUNK_SIZE):
        selected_chunk = thin_selected(
            selected,
            energy_indices=chunk,
            height_indices=height_indices,
        )

        atmosphere = propagate_atmosphere_coherent(
            selected_chunk,
            pmns,
            dm21,
            dm3l,
            detector_depth_m=DETECTOR_DEPTH_M,
            antinu=antinu,
            matter=MATTER_IN_atmosphere,
            atmosphere_n_steps=atmosphere_STEPS,
            trajectory_steps=TRAJECTORY_STEPS,
            device=device,
            dtype=dtype,
            debug=False,
        )

        earth = propagate_earth_coherent(
            atmosphere,
            pmns,
            dm21,
            dm3l,
            detector_depth_m=DETECTOR_DEPTH_M,
            density=density,
            antinu=antinu,
            reunitarize_earth=REUNITARIZE_earth,
            device=device,
            dtype=dtype,
            debug=False,
        )

        source_surface = integrate_initial_and_surface_fluxes(
            {selected_chunk["particle"]: selected_chunk},
            {selected_chunk["particle"]: atmosphere},
            device=device,
            dtype=dtype,
        )

        integrated = integrate_height_and_sum_flavours(
            {selected_chunk["particle"]: earth},
            device=device,
            dtype=dtype,
        )

        particle = selected_chunk["particle"]
        detector_flux = integrated["detector_flux_total_Ei"]
        initial_flux = source_surface["initial_flux_by_beta"][particle]
        surface_flux = source_surface["surface_flux_by_beta"][particle]
        surface_probability = source_surface["surface_probability_by_beta"][particle]
        source_flux = torch.sum(initial_flux, dim=-1)
        detector_probability = detector_flux / source_flux[..., None].clamp_min(
            torch.finfo(dtype).tiny
        )

        E_chunks.append(selected_chunk["E_grid_GeV"])
        detector_flux_chunks.append(detector_flux)
        surface_flux_chunks.append(surface_flux)
        initial_flux_chunks.append(initial_flux)
        detector_probability_chunks.append(detector_probability)
        surface_probability_chunks.append(surface_probability)
        source_flux_chunks.append(source_flux)

    return {
        "E_grid_GeV": torch.cat(E_chunks, dim=0),
        "detector_flux_Ei": torch.cat(detector_flux_chunks, dim=0),
        "surface_flux_Ei": torch.cat(surface_flux_chunks, dim=0),
        "initial_flux_Ei": torch.cat(initial_flux_chunks, dim=0),
        "detector_probability_Ei": torch.cat(detector_probability_chunks, dim=0),
        "surface_probability_Ei": torch.cat(surface_probability_chunks, dim=0),
        "source_flux_E": torch.cat(source_flux_chunks, dim=0),
    }


## 3. Main run function

`main()` coordinates the configured workflow: it validates parameters, loads or generates input data, dispatches the numerical computation, writes output files, and returns a compact summary object.


In [12]:
def main():
    load_device = resolve_device(LOAD_DEVICE)
    propagation_device = resolve_device(PROPAGATION_DEVICE)

    print("\nDetector flux propagation")
    print(f"Input directory      : {INPUT_DIR}")
    print(f"Output directory     : {OUTPUT_DIR}")
    print(f"earth density file   : {EARTH_DENSITY_FILE}")
    print(f"Load device          : {load_device}")
    print(f"Propagation device   : {propagation_device}")
    print(f"Energy chunk size    : {ENERGY_CHUNK_SIZE}")
    print(f"atmosphere steps     : {atmosphere_STEPS}")
    print(f"Trajectory steps     : {TRAJECTORY_STEPS}")

    flux_data = load_directory(
        INPUT_DIR,
        map_location="cpu",
        dtype=LOAD_DTYPE,
        device=load_device,
        group_by="particle",
        verbose=DEBUG,
    )

    particles = list(PARTICLES) if PARTICLES is not None else sorted(flux_data.keys())

    pmns = PMNS(
        THETA12,
        THETA13,
        THETA23,
        DELTA_CP,
        device=propagation_device,
        real_dtype=COMPUTE_DTYPE,
    )

    density = load_earth_density_from_csv(
        EARTH_DENSITY_FILE,
        device=propagation_device,
        dtype=COMPUTE_DTYPE,
    )

    dm21 = torch.as_tensor(DM21_EV2, device=propagation_device, dtype=COMPUTE_DTYPE)
    dm3l = torch.as_tensor(DM3L_EV2, device=propagation_device, dtype=COMPUTE_DTYPE)

    saved_paths = []
    t0 = time.perf_counter()

    for particle in particles:
        if particle not in flux_data:
            print(f"Skipping missing particle: {particle}")
            continue

        group = flux_data[particle]
        n_angles = int(group["theta_grid_deg"].numel())
        angle_indices = evenly_spaced_indices(n_angles, ANGLE_MAX_COUNT)
        energy_indices_full = evenly_spaced_indices(
            int(group["E_grid_GeV"].numel()),
            ENERGY_MAX_COUNT,
        )
        height_indices = evenly_spaced_indices(
            int(group["h_grid_km"].numel()),
            HEIGHT_MAX_COUNT,
        )

        for i_angle, angle_index in enumerate(angle_indices.tolist()):
            selected = select_particle_angle_flux(
                flux_data,
                particle,
                angle_index=angle_index,
                device=propagation_device,
                dtype=COMPUTE_DTYPE,
            )

            alpha_deg, theta_deg = angle_value(selected)
            output_path = build_detector_flux_path(
                OUTPUT_DIR,
                particle,
                alpha_deg=alpha_deg,
                theta_deg=theta_deg,
                base_filename=OUTPUT_FILENAME,
            )

            if SKIP_EXISTING and output_path and Path(output_path).exists() and not OVERWRITE:
                if DEBUG:
                    print(f"Skipping existing: {output_path}")
                saved_paths.append(output_path)
                continue

            if DEBUG:
                alpha_text = "None" if alpha_deg is None else f"{alpha_deg:.3f}"
                print(
                    f"\n{particle} | angle {i_angle + 1}/{angle_indices.numel()} "
                    f"| alpha={alpha_text} deg | theta={theta_deg:.3f} deg "
                    f"| antinu={infer_antinu(particle)}"
                )

            propagated = propagate_particle_angle(
                selected=selected,
                pmns=pmns,
                density=density,
                dm21=dm21,
                dm3l=dm3l,
                antinu=infer_antinu(particle),
                energy_indices=energy_indices_full,
                height_indices=height_indices,
                device=propagation_device,
                dtype=COMPUTE_DTYPE,
            )

            h_grid = _as_tensor(
                selected["h_grid_km"][height_indices],
                device=propagation_device,
                dtype=COMPUTE_DTYPE,
            )

            result = {
                "particle": particle,
                "antinu": torch.as_tensor(infer_antinu(particle)),
                "alpha_deg": None if alpha_deg is None else torch.as_tensor(alpha_deg),
                "theta_deg": torch.as_tensor(theta_deg),
                "E_grid_GeV": propagated["E_grid_GeV"],
                "h_grid_km": h_grid,
                "detector_flux_Ei": propagated["detector_flux_Ei"],
                "surface_flux_Ei": propagated["surface_flux_Ei"],
                "initial_flux_Ei": propagated["initial_flux_Ei"],
                "detector_probability_Ei": propagated["detector_probability_Ei"],
                "surface_probability_Ei": propagated["surface_probability_Ei"],
                "source_flux_E": propagated["source_flux_E"],
                "metadata_extra": {
                    "input_dir": INPUT_DIR,
                    "earth_density_file": EARTH_DENSITY_FILE,
                    "detector_depth_m": float(DETECTOR_DEPTH_M),
                    "matter_in_atmosphere": bool(MATTER_IN_atmosphere),
                    "reunitarize_earth": bool(REUNITARIZE_earth),
                    "atmosphere_steps": int(atmosphere_STEPS),
                    "trajectory_steps": int(TRAJECTORY_STEPS),
                    "energy_chunk_size": ENERGY_CHUNK_SIZE,
                    "energy_max_count": ENERGY_MAX_COUNT,
                    "height_max_count": HEIGHT_MAX_COUNT,
                    "angle_index": int(angle_index),
                    "pmns": {
                        "theta12": float(THETA12),
                        "theta13": float(THETA13),
                        "theta23": float(THETA23),
                        "delta": float(DELTA_CP),
                    },
                    "mass_splittings_ev2": {
                        "DeltamSq21": float(DM21_EV2),
                        "DeltamSq3l": float(DM3L_EV2),
                    },
                },
            }

            saved_path = save_detector_flux_result(
                result,
                OUTPUT_DIR,
                particle=particle,
                alpha_deg=alpha_deg,
                theta_deg=theta_deg,
                base_filename=OUTPUT_FILENAME,
                dtype=SAVE_DTYPE,
                overwrite=OVERWRITE,
            )
            saved_paths.append(saved_path)
            print(f"Saved: {saved_path}")

    elapsed = time.perf_counter() - t0
    print(f"\nFinished detector propagation. Files visited/saved: {len(saved_paths)}")
    print(f"Elapsed: {elapsed:.3f} s")

    return saved_paths


## 4. Execute run

The final cell launches the configured production run and stores the returned object in `RUN_RESULT`. Generation notebooks return the generation metadata and optional stacked tensors; propagation notebooks return the visited or saved detector-flux paths. Keep `DEBUG=True` while validating paths and reduce it for long production runs.


In [13]:
RUN_RESULT = main()
RUN_RESULT



Detector flux propagation
Input directory      : V:\output\data\atmosphere\mceq\diff_flux_QGSJETII04_PolyGonato_ICECUBE
Output directory     : V:\output\data\atmosphere\detector\detector_flux_QGSJETII04_PolyGonato_ICECUBE
earth density file   : G:\Mi unidad\03.Codigo\034.TFM.UV\Tpeanuts\data\density\earth_density.csv
Load device          : cpu
Propagation device   : cuda:0
Energy chunk size    : 8
atmosphere steps     : 120
Trajectory steps     : 120
Loaded antinue      | n_theta =  10 | phi(E,theta,h) shape = (10, 121, 501)
Loaded antinumu     | n_theta =  10 | phi(E,theta,h) shape = (10, 121, 501)
Loaded antinutau    | n_theta =  10 | phi(E,theta,h) shape = (10, 121, 501)
Loaded nue          | n_theta =  10 | phi(E,theta,h) shape = (10, 121, 501)
Loaded numu         | n_theta =  10 | phi(E,theta,h) shape = (10, 121, 501)
Loaded nutau        | n_theta =  10 | phi(E,theta,h) shape = (10, 121, 501)

antinue | angle 1/10 | alpha=None deg | theta=18.195 deg | antinu=True
Saved: V:\output

['V:\\output\\data\\atmosphere\\detector\\detector_flux_QGSJETII04_PolyGonato_ICECUBE\\detector_flux_antinue_theta_18p195deg.pt',
 'V:\\output\\data\\atmosphere\\detector\\detector_flux_QGSJETII04_PolyGonato_ICECUBE\\detector_flux_antinue_theta_31p788deg.pt',
 'V:\\output\\data\\atmosphere\\detector\\detector_flux_QGSJETII04_PolyGonato_ICECUBE\\detector_flux_antinue_theta_41p410deg.pt',
 'V:\\output\\data\\atmosphere\\detector\\detector_flux_QGSJETII04_PolyGonato_ICECUBE\\detector_flux_antinue_theta_49p458deg.pt',
 'V:\\output\\data\\atmosphere\\detector\\detector_flux_QGSJETII04_PolyGonato_ICECUBE\\detector_flux_antinue_theta_56p633deg.pt',
 'V:\\output\\data\\atmosphere\\detector\\detector_flux_QGSJETII04_PolyGonato_ICECUBE\\detector_flux_antinue_theta_63p256deg.pt',
 'V:\\output\\data\\atmosphere\\detector\\detector_flux_QGSJETII04_PolyGonato_ICECUBE\\detector_flux_antinue_theta_69p513deg.pt',
 'V:\\output\\data\\atmosphere\\detector\\detector_flux_QGSJETII04_PolyGonato_ICECUBE\\det